<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/GEM_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# @title Dependencies

# first-party
from typing import Dict, Any, Optional, Tuple, List
import random
import time
import csv
import os

# third-party
from google.colab import drive
import pandas as pd
from openai import OpenAI





In [2]:
# @title Paths and definitions

# mount GDrive
drive.mount('/content/drive')

# dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"
# create dataset directory (if not already existing)
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# raw directory
RAW_DIR = "/content/drive/MyDrive/Thesis/raw/"
# create raw directory (if not already existing)
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# define result directory
RESULTS_DIR = RAW_DIR # Using RAW_DIR as RESULTS_DIR is the same path

# define path directory
LOG_DIR = RAW_DIR # Using RAW_DIR as LOG_DIR is the same path

# system prompt for judgement processing
SYSTEM_PROMPT_PROCESSING = """

Carefully read the text of a scientific paper review. You should summarize each evaluation in the review in a separate line.
Begin each summary line with one of the following phrases: ’The reviewer appreciates’, ’The reviewer criticizes’, ’The reviewer questions’, ’The reviewer suggests’.
You need to keep the summary as concise as possible, excluding specific details about the paper’s content, such as topics, ideas, methods, findings, and any mathematical symbols.
You should ensure that even if multiple evaluations are mentioned in the same sentence in the original review, you should still split it into separate lines.
For example, you should not output a line like ’The reviewer appreciates the well-written paper and good experimental performance’.
In contrast, you should output ’The reviewer appreciates the well-written paper’ and ’The reviewer appreciates good experimental performance’ in two lines.

"""

OPENAI_KEY = "Set key here"

SIMULATE = True  # Set to False to make real API calls

MAX_RETRIES = 3

RETRY_DELAY = 2.0

PREPROCESSING_MODEL = "gpt-4o-2024-05-13"

# paper data + human reviews
DATASET_PAPER = "dataset_paper.parquet"

# llm reviews
LLM_REVIEW = 'llm_sim_results.csv'

# result file
STAND_REVW_CSV = "standardise_human_review_1_res.csv"

# log file
LOG_FILE_CSV = "standardise_human_review_1_log.csv"


Mounted at /content/drive


In [3]:
# @title Load ICLR data

# full path
FULL_PATH_ICLR = DATASET_DIR + DATASET_PAPER

try:
    # Load the Parquet file into a Pandas DataFrame
    df_reviews_source = pd.read_parquet(FULL_PATH_ICLR)

    # check
    print("\nDataFrame Columns:")
    print(df_reviews_source.columns.tolist())

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_ICLR}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")


DataFrame Columns:
['paper_id', 'pdf_url', 'abstract', 'parsed_text', 'human_review1', 'human_review2', 'human_review3']


In [4]:
# @title Load LLM reviews

# full path
FULL_PATH_LLM = RAW_DIR + LLM_REVIEW

# adjust data types of directory
LLM_CSV_DTYPES = {
    'document_id': str,
    'review_text': str,
    'raw_api_output': str,
    'model': str,
    'temperature': float,
    'reasoning_effort': str,
    'chain_of_thought': str,
    'CoT': str
}

try:
    # Load the CSV file into a Pandas DataFrame
    df_llm_results = pd.read_csv(FULL_PATH_LLM, dtype=LLM_CSV_DTYPES)

    # check
    print("\nDataFrame Head:")
    print(df_llm_results.head())

    print("\nDataFrame Columns:")
    print(df_llm_results.columns)

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_LLM}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")



DataFrame Head:
   document_id                                        review_text  \
0  vuD2xEtxZcj  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...   
1  WoByU5W5te0  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...   
2  XZRmNjUMj0c  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...   
3  cDYRS5iZ16f  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...   
4  Sh97TNO5YY_  {\n  "model": "gpt-5-mini-2025-08-07",\n  "tem...   

                   model  temperature reasoning_effort chain_of_thought  
0  gpt-5-mini-2025-08-07          NaN              low              NaN  
1  gpt-5-mini-2025-08-07          NaN              low              NaN  
2  gpt-5-mini-2025-08-07          NaN              low              NaN  
3  gpt-5-mini-2025-08-07          NaN              low              NaN  
4  gpt-5-mini-2025-08-07          NaN              low              NaN  

DataFrame Columns:
Index(['document_id', 'review_text', 'model', 'temperature',
       'reasoning_effort', 'chain_of_though

In [5]:
# @title Merging

try:
    # full-join merge
    df_merged_full = pd.merge(
        left=df_reviews_source,
        right=df_llm_results,
        left_on='paper_id',
        right_on='document_id',
        how='outer'
    )

    # check
    print(df_merged_full.head(57))

    # check
    print(df_merged_full.columns.tolist())

# error message
except Exception as e:
    print(f"An error occured while merging the files: {e}")


       paper_id                                            pdf_url  \
0    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
1    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
2    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
3    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
4    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
5    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
6    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
7    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
8    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
9    -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
10   -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
11   -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
12   -0tPmzgXS5  https://openreview.net/pdf/d693e0ef8d0dfd32267...   
13   -0tPmzgXS5  htt

# Preprocessing

In [6]:
# @title Define simulation

# define LLMClient
class LLMClient:
    """
    Handles API calls, simulation, and retry logic (adopted from supervisor's pattern).
    """
    def __init__(self, simulate: bool = True, api_key: str = OPENAI_KEY, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self._openai = None
        self.api_key = api_key

        # Initialize client if not simulating
        self._maybe_init_clients()

    def _maybe_init_clients(self):
        """Initializes the real OpenAI client if not in simulation mode."""
        if self.simulate:
            print("INFO: LLMClient initialized in SIMULATION mode.")
            return

        if self.api_key == "SK-mock-key":
             print("WARNING: Using mock API key. Real API calls will likely fail. Please set OPENAI_API_KEY.")

        try:
            # We only need the OpenAI client for your current use case
            self._openai = OpenAI(api_key=self.api_key)
            print("INFO: LLMClient initialized for REAL API calls.")
        except Exception as e:
            print(f"ERROR: Could not initialize OpenAI client: {e}")
            self.simulate = True # Fallback to simulation if initialization fails

    def call(
        self,
        system_prompt: str,
        user_prompt: str,
        temperature: float,
        max_tokens: int,
        max_retries: int = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        model: str = PREPROCESSING_MODEL
        ) -> Tuple[str, str]:
        """
        Executes the API call with retry logic (adopted from supervisor's pattern).

        Returns: (raw_text, status_code/error_message)
        """

        if self.simulate:
            # --- SIMULATION PATH (quick return) ---
            simulated_text = (
                f"Simulated Standardization: The core argument was '{user_prompt[:50]}...'. "
                f"Model={model}, Temp={temperature}. Output is a concise summary."
            )
            return simulated_text, "SIMULATED"

        # --- REAL API PATH (with retry loop) ---
        for attempt in range(1, max_retries + 1):
            try:
                # Direct implementation of your original function logic
                response = self._openai.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                # Success: Extract the text and return
                generated_text = response.choices[0].message.content
                return generated_text, "SUCCESS"

            except Exception as e:
                error_msg = f"API Error (Attempt {attempt}/{max_retries}): {type(e).__name__}: {e}"

                if attempt == max_retries:
                    print(f"FATAL: {error_msg}")
                    return f"ERROR: Failed after {max_retries} attempts. {e}", "FAILURE"

                print(f"WARNING: {error_msg}. Retrying in {retry_delay}s...")
                time.sleep(retry_delay)

        return "ERROR: Loop finished without success.", "FAILURE" # Should not be reached


# define review call
def standardise_review(
    client: LLMClient,
    system_prompt_processing: str,
    user_prompt_processing: str,
    id_variable: str,
    model_from_row: str,
    temperature_from_row: float,
    reasoning_effort_from_row: Optional[str] = None,
    chain_of_thought_from_row: Optional[str] = None,
    temperature: float = 0,
    max_out_tokens: int = 4000,
    model: str = PREPROCESSING_MODEL
) -> pd.DataFrame:

    # use the LLMClient's call method
    generated_text, status = client.call(
        model=model,
        system_prompt=system_prompt_processing,
        user_prompt=user_prompt_processing,
        temperature=temperature,
        max_tokens=max_out_tokens,
    )

    # initialize list
    records = []

    # append results to the list
    records.append({
        'paper_id': id_variable,
        'standardised_review_text': generated_text,
        'standardisation_status': status,
        'error_message': None,
        'processed_timestamp': None,
        'model': model_from_row,
        'temperature': temperature_from_row,
        'reasoning_effort': reasoning_effort_from_row,
        'chain_of_thought': chain_of_thought_from_row
    })

    return records

In [ ]:
# @title Run simulation

# define paths for results and log files
RESULTS_PATH = RESULTS_DIR + STAND_REVW_CSV
LOG_PATH = LOG_DIR + LOG_FILE_CSV

# initialize the client
client = LLMClient(simulate=SIMULATE)

# define columns for the result file
STAND_REVW_COLMNS = ['paper_id',
                 'standardised_review_text',
                 "standardisation_status",
                 "error_message",
                 "processed_timestamp",
                 'model',
                 'temperature',
                 'reasoning_effort',
                 'chain_of_thought']

# check file existence
if not os.path.exists(RESULTS_PATH):
  with open(RESULTS_PATH, 'w', newline='') as f:
      writer = csv.writer(f)
      writer.writerow(STAND_REVW_COLMNS)

# define columns for the log file
LOG_COLUMNS = ['model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'papers_processed_count', 'papers_succeeded_count', 'configuration_status']

# check file existence
if not os.path.exists(LOG_PATH):
    with open(LOG_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(LOG_COLUMNS)

# copy merged dataframe
df_merged_full_grouped = df_merged_full.copy()

# filling NaNs and Nones to placeholders and format columns
df_merged_full_grouped['temperature_group_key'] = df_merged_full_grouped['temperature'].fillna('NaN_temp').astype(str)
df_merged_full_grouped['reasoning_effort_group_key'] = df_merged_full_grouped['reasoning_effort'].fillna('NaN_effort').astype(str)
df_merged_full_grouped['chain_of_thought_group_key'] = df_merged_full_grouped['chain_of_thought'].fillna('NaN_CoT').astype(str)

# group by the unique configuration parameters
config_groups = df_merged_full_grouped.groupby(['model', 'temperature_group_key', 'reasoning_effort_group_key', 'chain_of_thought_group_key'])

# iterate over each unique configuration
for config_key, config_group_df in config_groups:
    current_model_key, current_temperature_key, current_reasoning_effort_key, current_chain_of_thought_key = config_key

    # extract original values for logging from a sample row in the group
    sample_row = config_group_df.iloc[0]
    original_model_for_log = sample_row['model']
    original_temperature_for_log = sample_row['temperature']
    original_reasoning_effort_for_log = sample_row['reasoning_effort']
    original_chain_of_thought_for_log = sample_row['chain_of_thought']

    # count of total papers in config
    papers_in_config = len(config_group_df)

    # initial count of successfull papers in config
    successful_papers_in_config = 0

    # iterate over each paper within this configuration group
    for i, row in config_group_df.iterrows():

        # content for the client call
        review = row['human_review1']

        # configurational settings for the result file
        paper_id = row['paper_id']
        model_from_row = row['model']
        temperature_from_row = row['temperature']
        reasoning_effort_from_row = row['reasoning_effort']
        chain_of_thought_from_row = row['chain_of_thought']

        # Itterate over attempts (in case of previous failure)
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                # apply the call-function
                df_result = standardise_review(
                                  client=client,
                                  system_prompt_processing=SYSTEM_PROMPT_PROCESSING,
                                  user_prompt_processing=review,
                                  id_variable=paper_id,
                                  model_from_row=model_from_row,
                                  temperature_from_row=temperature_from_row,
                                  reasoning_effort_from_row=reasoning_effort_from_row,
                                  chain_of_thought_from_row=chain_of_thought_from_row
                                  )

                # convert the result to a dataframe
                df_result = pd.DataFrame(df_result)

                # ensure correct column initialisation
                df_result = df_result[STAND_REVW_COLMNS]

                # always append to the results file (header already handled)
                df_result.to_csv(RESULTS_PATH, mode="a", header=False, index=False)

                # information message of success
                print(f"[SUCCESS] Processed {paper_id} with config ({original_model_for_log}, {original_temperature_for_log}, {original_reasoning_effort_for_log}, {original_chain_of_thought_for_log}) on attempt {attempt}/{MAX_RETRIES}.")

                # increase success tracker
                successful_papers_in_config += 1

                # stop loop in case of success
                break

            # in case of failure
            except Exception as e:
                # if MAX_RETRIES not reached yet
                if attempt < MAX_RETRIES:

                    #calculate delay
                    delay = RETRY_DELAY * (2 ** (attempt - 1))

                    # information message of retry
                    print(f"[RETRY] Paper {paper_id} with config ({original_model_for_log}, {original_temperature_for_log}, {original_reasoning_effort_for_log}, {original_chain_of_thought_for_log}) failed (Attempt {attempt}/{MAX_RETRIES}). Retrying in {delay:.1f}s. Error: {type(e).__name__}: {e}")

                    # delay next call
                    time.sleep(delay)

                    # retry
                    continue

                # if MAX_RETRIES already reached
                else:
                    # setup artifiical report with error information
                    failure_record = {
                        'paper_id': paper_id,
                        'standardised_review_text': None,
                        'standardisation_status': "FAILURE",
                        'error_message': f"Permanent failure after {MAX_RETRIES} attempts. Last Error: {type(e).__name__}: {str(e)}",
                        'processed_timestamp': time.time(),
                        'model': model_from_row,
                        'temperature': temperature_from_row,
                        'reasoning_effort': reasoning_effort_from_row,
                        'chain_of_thought': chain_of_thought_from_row
                    }

                    # convert the result to a dataframe
                    df_result = pd.DataFrame([failure_record])

                    # ensure correct column initialisation
                    df_result = df_result[STAND_REVW_COLMNS]

                    # always append to the results file
                    df_result.to_csv(RESULTS_PATH, mode="a", header=False, index=False)

                    # information messsage of permanent failure
                    print(f"[FAILURE] Permanently failed to process {paper_id} with config ({original_model_for_log}, {original_temperature_for_log}, {original_reasoning_effort_for_log}, {original_chain_of_thought_for_log}) after {MAX_RETRIES} attempts.")

                    # stop loop after final unseccesful attempt
                    break

    # capture log all log information
    log_entry = {
        'model': original_model_for_log,
        'temperature': original_temperature_for_log,
        'reasoning_effort': original_reasoning_effort_for_log,
        'chain_of_thought': original_chain_of_thought_for_log,
        'papers_processed_count': papers_in_config,
        'papers_succeeded_count': successful_papers_in_config,
        'configuration_status': "SUCCESS" if successful_papers_in_config == papers_in_config else "PARTIAL_FAILURE" if successful_papers_in_config > 0 else "TOTAL_FAILURE"
    }

    # convert the log_entry to a dataframe
    log_df = pd.DataFrame([log_entry])

    # append to the log file
    log_df[LOG_COLUMNS].to_csv(LOG_PATH, mode="a", header=False, index=False)

    # information message for this parameter combination
    print(f"Logged status for configuration ({original_model_for_log}, {original_temperature_for_log}, {original_reasoning_effort_for_log}, {original_chain_of_thought_for_log}): {log_entry['configuration_status']} (Processed {successful_papers_in_config}/{papers_in_config} papers).")


In [ ]:
# @title Client Setup

# function for the simple API call
def standardise_review(system_prompt_processing: str, user_prompt_processing: str, id_variable: str, temperature: float = 0, max_out_tokens: int = 4000, model: str = "gpt-4o-2024-05-13"): # defaults as Xu et al.
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt_processing},
                {"role": "user", "content": user_prompt_processing}
            ],
            temperature=temperature,
            max_tokens=max_out_tokens
        )

        # Extract the review text from the response
        generated_text = response.choices[0].message.content
        return {
            'id': id_variable,
            'standardised_review_text': generated_text
        }

    except Exception as e:
        print(f"An API error occurred: {e}")
        return {'id': id_variable,
                'standardised_review_text': f"An API error occurred: {e}"
                }

## Summary of author stated strengths and weaknesses

**System prompt**

You are given a paper submission for a top-tier Machine Learning conference. Your goal is to identify and list the strengths and weaknesses that the paper claims about itself. This task requires careful reading of the paper. Please follow these steps to complete the task: 1. Carefully read the entire paper submission. As you read, identify instances where the authors mention strengths or positive aspects of their research, methodology, results, or contributions. These are the strengths claimed by the paper. Also, identify instances where the authors mention limitations, weaknesses, or areas for future improvement in their work. These are the weaknesses claimed by the paper. 2. Compile your findings into two separate lists: one for strengths and one for weaknesses. 3. For each list, write each point on a separate line, keeping it concise. Add an extra blank line between each point for clarity. 4. Format your output as follows: ¡strengths claimed by the paper¿ [List each strength claimed by the paper in separate lines, with an extra blank line between each point] ¡/strengths claimed by the paper¿ ¡weaknesses claimed by the paper¿ [List each weakness claimed by the paper in separate lines, with an extra blank line between each point] ¡/weaknesses claimed by the paper¿ Important: Focus only on the strengths and weaknesses that the paper claims about itself. Do not include your own evaluation or opinion of the paper’s merits or shortcomings. Do not include the strengths and weaknesses of the baseline. Your task is to report what the authors themselves have stated about their work’s strengths and limitations.

**User prompt**

{Full paper in Text Format}

## Judgement prediction

**System Prompt**

You are the second reviewer for a scientific paper. You are given the abstract of the paper and a list of review judgments from the first reviewer, starting with ’The reviewer appreciates/criticizes/questions/suggests’. Your task is to provide your own judgments of the paper based on the given materials. You should create a separate line for each judgment you have, starting with ’The reviewer appreciates/criticizes/questions/suggests’. Ensure your judgments are concise, excluding specific details about the paper’s content.

**User Prompt**

[Abstract of the paper]

{Abstract of the Paper if the metric is GEM-S, and “Not Available” if the metric is GEM}

[Review judgments from the first reviewer]

{Preprocessed Review Judgments (x) for conditional probability, and “Not Available” for marginal probability}

**Forced LLM Output**

{Preprocessed Review Judgments (y)}